In [ ]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from dataset_configs.smpl.utils import compute_B_from_beta, smpl_pose_to_D_orientation
from pipeline.evaluate import evaluate, print_metrics, save_metrics
from wimusim import WIMUSim, utils

import pathlib
pathlib.Path("results").mkdir(exist_ok=True)

## Evaluation: Virtual vs Real IMU

This notebook evaluates WIMUSim's simulation accuracy on **TotalCapture** and **EMDB** test sets.

For each test sequence:
1. Load SMPL parameters → initialize B and D
2. Load real IMU data → target
3. Run WIMUSim → virtual IMU
4. Compute metrics per IMU

| Metric | Unit | Meaning |
|--------|------|---------|
| RMSE | m/s² / rad/s | Error magnitude |
| MAE  | m/s² / rad/s | Average absolute error |
| PCC  | — | Waveform similarity (1.0 = perfect) |

## 1. Evaluate on TotalCapture

In [ ]:
from dataset_configs.totalcapture import consts as tc_consts
from dataset_configs.totalcapture.utils import (
    load_smpl_params, load_imu_data, generate_default_placement_params,
)

TOTALCAPTURE_PATH = "path/to/totalcapture"
SMPL_MODEL_PATH   = "path/to/smpl/models"

tc_results_all = []

for subject in tc_consts.TEST_SUBJECTS:
    for activity in tc_consts.ACTIVITY_LIST:
        print(f"  {subject}/{activity} ...", end=" ")
        try:
            betas, global_orient, body_pose = load_smpl_params(
                TOTALCAPTURE_PATH, subject, activity
            )
            real_imu = load_imu_data(TOTALCAPTURE_PATH, subject, activity)
        except FileNotFoundError:
            print("skipped")
            continue

        B_rp        = compute_B_from_beta(betas, SMPL_MODEL_PATH)
        orientation = smpl_pose_to_D_orientation(global_orient, body_pose)
        P_params    = generate_default_placement_params(B_rp)

        env = WIMUSim(
            B={"rp": B_rp},
            D={"orientation": orientation, "sample_rate": tc_consts.SMPL_SAMPLE_RATE},
            P=P_params,
            H=utils.generate_default_H_configs(tc_consts.IMU_NAMES),
            dataset_name="SMPL",
            device="cpu",
        )
        virt_imu = env.simulate(mode="generate")

        df = evaluate(virt_imu, real_imu)
        df["subject"]  = subject
        df["activity"] = activity
        tc_results_all.append(df)
        print("done")

In [ ]:
tc_df = pd.concat(tc_results_all, ignore_index=True)

tc_summary = (
    tc_df[tc_df["imu"] == "MEAN"]
    .groupby("modality")[["rmse", "mae", "pearson"]]
    .mean()
)
print("=== TotalCapture Summary ===")
print(tc_summary.round(4))

save_metrics(tc_df, "results/totalcapture_metrics.csv")

## 2. Evaluate on EMDB

In [ ]:
from dataset_configs.emdb import consts as emdb_consts
from dataset_configs.emdb.utils import (
    load_smpl_params as emdb_load_smpl,
    load_imu_data    as emdb_load_imu,
    iter_sequences,
    generate_default_placement_params as emdb_placement,
)

EMDB_PATH = "path/to/emdb"

emdb_results_all = []

for subject, seq_name, pkl_path in iter_sequences(EMDB_PATH, split=emdb_consts.TEST_SPLIT):
    print(f"  {subject}/{seq_name} ...", end=" ")
    try:
        betas, global_orient, body_pose = emdb_load_smpl(pkl_path)
        real_imu = emdb_load_imu(pkl_path)
    except Exception as e:
        print(f"skipped ({e})")
        continue

    B_rp        = compute_B_from_beta(betas, SMPL_MODEL_PATH)
    orientation = smpl_pose_to_D_orientation(global_orient, body_pose)
    P_params    = emdb_placement(B_rp)

    env = WIMUSim(
        B={"rp": B_rp},
        D={"orientation": orientation, "sample_rate": emdb_consts.SMPL_SAMPLE_RATE},
        P=P_params,
        H=utils.generate_default_H_configs(emdb_consts.IMU_NAMES),
        dataset_name="SMPL",
        device="cpu",
    )
    virt_imu = env.simulate(mode="generate")

    df = evaluate(virt_imu, real_imu)
    df["subject"]  = subject
    df["sequence"] = seq_name
    emdb_results_all.append(df)
    print("done")

In [ ]:
emdb_df = pd.concat(emdb_results_all, ignore_index=True)

emdb_summary = (
    emdb_df[emdb_df["imu"] == "MEAN"]
    .groupby("modality")[["rmse", "mae", "pearson"]]
    .mean()
)
print("=== EMDB Summary ===")
print(emdb_summary.round(4))

save_metrics(emdb_df, "results/emdb_metrics.csv")

## 3. Visualize: Virtual vs Real Waveform

Compare one sequence side-by-side to check waveform alignment.

In [ ]:
imu_name = "LLA"   # change to any IMU present in the last evaluated sequence
start, end = 0, 300

acc_virt,  gyro_virt  = virt_imu[imu_name]
acc_real,  gyro_real  = real_imu[imu_name]

acc_virt  = acc_virt.detach().cpu().numpy()
gyro_virt = gyro_virt.detach().cpu().numpy()
acc_real  = np.asarray(acc_real)
gyro_real = np.asarray(gyro_real)

fig, axs = plt.subplots(3, 2, figsize=(14, 7))
fig.suptitle(f"Virtual vs Real IMU  ({imu_name})", fontsize=13)
axs[0, 0].set_title("Accelerometer (m/s²)")
axs[0, 1].set_title("Gyroscope (rad/s)")

for i, axis in enumerate(["X", "Y", "Z"]):
    axs[i, 0].plot(acc_real[start:end,  i], label="Real",    alpha=0.8)
    axs[i, 0].plot(acc_virt[start:end,  i], label="Virtual", alpha=0.8, linestyle="--")
    axs[i, 0].set_ylabel(axis)
    axs[i, 0].legend(fontsize=8)

    axs[i, 1].plot(gyro_real[start:end, i], label="Real",    alpha=0.8)
    axs[i, 1].plot(gyro_virt[start:end, i], label="Virtual", alpha=0.8, linestyle="--")
    axs[i, 1].set_ylabel(axis)
    axs[i, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("results/waveform_comparison.png", dpi=150)
plt.show()

In [ ]:
# Per-IMU RMSE bar chart — accelerometer, EMDB
per_imu = (
    emdb_df[(emdb_df["modality"] == "acc") & (emdb_df["imu"] != "MEAN")]
    .groupby("imu")["rmse"].mean()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 4))
per_imu.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("EMDB — Accelerometer RMSE per IMU (m/s²)")
ax.set_ylabel("RMSE")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("results/per_imu_rmse.png", dpi=150)
plt.show()